VGGMM: 성분 병합 제안 → 전체 재적합 → 동일모형 ELBO로 채택

독립 실행 파일. VS Code에서 # %% 셀을 위에서부터 실행하세요.
필요한 패키지: numpy scipy pandas matplotlib scikit-learn threadpoolctl.
초기 shape 기본값은 앞선 대조 실험에서 더 나았던 moment입니다.
merge on/off 비교는 동일한 초기 fit을 공유합니다. shape는 계속 자유롭게 추정합니다.
모든 단계에서 Kmax와 Normal/Gamma/Beta prior가 그대로 유지됩니다.
책임도 두 열을 합치고 하나를 비우되, 비운 슬롯도 E-step에 참여합니다.
모델의 슬롯 순서를 재정렬하지 않으며, 각 후보는 복사된 상태에서 최적화합니다.
전체 ELBO는 likelihood + z entropy - stick/Normal/Gamma KL입니다.
Truth는 생성·사후평가에만 사용하고, fit/병합/채택에는 입력하지 않습니다.
IEEE 9002852의 finite/Taylor 알고리즘을 그대로 재현한 구현은 아닙니다.
독립 truncated-DP variational GGMM, b point estimate, plug-in density입니다.


In [ ]:
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import warnings
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import leggauss
from scipy.optimize import brentq, minimize
from scipy.special import betaln, digamma, gammaln, hyp1f1, logsumexp, xlogy
from scipy.stats import gennorm
from sklearn.mixture import BayesianGaussianMixture
from threadpoolctl import threadpool_limits

N = 5000
SAMPLE_SEED = 20260912
CAPS = (10, 11, 12, 13, 14, 15)
INITIAL_SHAPE_MODE = "moment"
INCLUDE_VGM = True


@dataclass(frozen=True)
class VGGMMConfig:
    max_components: int = 10
    concentration: float = .001
    active_threshold: float = .005
    max_iter: int = 300
    tol: float = 1e-5  # absolute ELBO change per observation, 3 consecutive times
    inner_max_iter: int = 30
    inner_ftol: float = 1e-8
    mean_prior_sd: float = 3.
    precision_prior_shape: float = 1.
    precision_prior_rate: float = .001
    starts: tuple[str, ...] = ("quantile", "lloyd")
    initial_shape_mode: str = "moment"
    enable_merge: bool = True
    max_merge_rounds: int = 15
    max_candidates_per_round: int = 3
    merge_gain_per_observation: float = 1e-4  # 10 * default outer tol
    candidate_min_effective_count: float = 1e-6


BASE_CONFIG = VGGMMConfig()


In [ ]:
@dataclass(init=False)
class Mixture:
    weights: np.ndarray
    means: np.ndarray
    scales: np.ndarray
    shapes: np.ndarray

    def __init__(self, weights, means, scales, shapes):
        for name, values in zip(("weights", "means", "scales", "shapes"),
                                (weights, means, scales, shapes)):
            setattr(self, name, np.array(values, dtype=float, copy=True))
        if (self.weights.ndim != 1 or not self.weights.size
            or any(getattr(self, name).shape != self.weights.shape
                   for name in ("means", "scales", "shapes"))
            or not all(np.isfinite(getattr(self, name)).all()
                       for name in ("weights", "means", "scales", "shapes"))
            or np.any(self.weights < 0) or not np.isclose(self.weights.sum(), 1.)
            or np.any(self.scales <= 0) or np.any(self.shapes <= 0)):
            raise ValueError("Invalid mixture parameters.")

    @property
    def k(self):
        return len(self.weights)

    @property
    def sigmas(self):
        return self.scales * np.exp(.5 * (gammaln(3/self.shapes) - gammaln(1/self.shapes)))

    def component_logpdf(self, x):
        return gennorm.logpdf(np.asarray(x).reshape(-1, 1), self.shapes,
                              loc=self.means, scale=self.scales)

    def logpdf(self, x):
        with np.errstate(divide="ignore"):
            return logsumexp(self.component_logpdf(x) + np.log(self.weights), axis=1)

    def pdf(self, x):
        return np.exp(self.logpdf(x))

    def cdf(self, x):
        return (gennorm.cdf(np.asarray(x).reshape(-1, 1), self.shapes,
                           loc=self.means, scale=self.scales) * self.weights).sum(axis=1)

    def payload(self):
        return {name: getattr(self, name).tolist()
                for name in ("weights", "means", "scales", "shapes")}


def sample_mixture(truth, n, seed):
    # Preserve the sampling order in the supplied implementation.
    rng = np.random.default_rng(seed)
    labels = rng.choice(truth.k, size=n, p=truth.weights)
    return np.asarray(gennorm.rvs(truth.shapes[labels], loc=truth.means[labels],
                      scale=truth.scales[labels], random_state=rng), dtype=float)


truth = Mixture(weights=[.2, .3, .5], means=[-5., 0., 6.],
                scales=[1.2, 1.6, 2.], shapes=[1.5, 4., 8.])
X = sample_mixture(truth, N, SAMPLE_SEED)
X.setflags(write=False)
SAMPLE_SHA256 = hashlib.sha256(np.asarray(X, dtype="<f8").tobytes()).hexdigest()
print(f"N={len(X):,}; seed={SAMPLE_SEED}; mean={X.mean():.9f}; sd={X.std():.9f}")


In [ ]:
def normal_log_abs_moment(x, mean, log_sd, b):
    """log E|x-U|**b, U ~ Normal(mean, exp(2*log_sd)); exact identity."""
    with np.errstate(over="raise", invalid="raise", divide="raise", under="ignore"):
        if not np.isfinite(b) or b <= 0 or not np.isfinite(log_sd):
            raise FloatingPointError("Invalid shape or posterior location SD.")
        sd = np.exp(log_sd)
        if sd == 0:
            raise FloatingPointError("Posterior location SD underflow.")
        z = -.5 * ((np.asarray(x) - mean)/sd)**2
        h = hyp1f1(-b/2, .5, z)
        if np.any(h <= 0) or not np.isfinite(h).all():
            raise FloatingPointError("Normal absolute moment cannot be represented.")
        answer = (b*log_sd + b*.5*np.log(2.) + gammaln((b+1)/2)
                  - .5*np.log(np.pi) + np.log(h))
    if not np.isfinite(answer).all():
        raise FloatingPointError("Nonfinite normal absolute moment.")
    return answer


def normal_kl(mean, log_sd, prior_sd):
    with np.errstate(over="raise", invalid="raise", under="ignore"):
        return .5*((np.exp(2*log_sd)+mean*mean)/prior_sd**2 - 1
                   + 2*np.log(prior_sd) - 2*log_sd)


def component_profile(y, x, r, cfg):
    """Optimize q(tau) analytically for fixed q(mu), b and responsibilities."""
    with np.errstate(over="raise", invalid="raise", divide="raise", under="ignore"):
        m, ls, b = float(y[0]), float(y[1]), float(np.exp(y[2]))
        lm = normal_log_abs_moment(x, m, ls, b)
        nk = float(r.sum())
        log_r = np.full_like(r, -np.inf)
        np.log(r, out=log_r, where=r > 0)
        log_s = float(logsumexp(log_r + lm))
        a0, rate0 = cfg.precision_prior_shape, cfg.precision_prior_rate
        a = a0 + nk/b
        lb = float(np.logaddexp(np.log(rate0), log_s))
        value = (nk*(-np.log(2.) - gammaln(1+1/b)) + gammaln(a)-gammaln(a0)
                 + a0*np.log(rate0) - a*lb - normal_kl(m, ls, cfg.mean_prior_sd))
    if not np.isfinite(value) or not np.isfinite(a):
        raise FloatingPointError("Nonfinite profiled ELBO.")
    return float(value), float(a), lb, b


def stick_posterior(nk, alpha):
    nk = np.asarray(nk, dtype=float)
    return 1+nk[:-1], alpha+np.cumsum(nk[::-1])[::-1][1:]


def stick_statistics(a, b):
    if len(a) == 0:
        return np.ones(1), np.zeros(1)
    elogv = digamma(a)-digamma(a+b)
    elog1v = digamma(b)-digamma(a+b)
    elogw = np.r_[elogv, 0.] + np.r_[0., np.cumsum(elog1v)]
    log_residual = np.log(b)-np.log(a+b)
    logw = np.r_[np.log(a)-np.log(a+b), 0.] + np.r_[0., np.cumsum(log_residual)]
    w = np.exp(logw)
    return w/w.sum(), elogw


def stick_kl(a, b, alpha):
    return float(np.sum(betaln(1., alpha)-betaln(a, b)+(a-1)*digamma(a)
        +(b-alpha)*digamma(b)-(a+b-1-alpha)*digamma(a+b)))


def gamma_kl(a, log_rate, a0, rate0):
    return ((a-a0)*digamma(a)-gammaln(a)+gammaln(a0)
            +a0*(log_rate-np.log(rate0))-a+rate0*a*np.exp(-log_rate))


def expected_log_components(x, y, gamma_a, gamma_log_rate):
    out = np.empty((len(x), len(y)))
    for k, row in enumerate(y):
        b = float(np.exp(row[2]))
        lm = normal_log_abs_moment(x, row[0], row[1], b)
        with np.errstate(over="ignore", invalid="raise"):
            out[:, k] = (-np.log(2.)-gammaln(1+1/b)
                +(digamma(gamma_a[k])-gamma_log_rate[k])/b
                -np.exp(np.log(gamma_a[k])-gamma_log_rate[k]+lm))
    return out


def elbo_value(r, expected_logpdf, sa, sb, y, ga, glb, cfg):
    _, elogw = stick_statistics(sa, sb)
    term = np.zeros_like(r)
    np.multiply(r, expected_logpdf+elogw, out=term, where=r > 0)
    value = term.sum()-xlogy(r, r).sum()-stick_kl(sa, sb, cfg.concentration)
    value -= sum(normal_kl(row[0], row[1], cfg.mean_prior_sd) for row in y)
    value -= gamma_kl(ga, glb, cfg.precision_prior_shape, cfg.precision_prior_rate).sum()
    return float(value)


def moment_initial_shape(group):
    sd = float(np.std(group))
    ratio = float(np.mean(np.abs(group-group.mean()))/sd) if sd > 0 else np.nan
    if len(group) < 4 or not 0 < ratio < np.sqrt(3)/2:
        return 2.

    def f(eta):
        q = np.exp(-eta)
        return (.5*np.log(3)-np.log(2)+gammaln(1+2*q)
                -.5*(gammaln(1+q)+gammaln(1+3*q))-np.log(ratio))

    lo, hi = -1., 1.
    while f(lo) > 0 and lo > -16:
        lo -= 1
    while f(hi) < 0 and hi < 24:
        hi += 1
    return float(np.exp(brentq(f, lo, hi))) if f(lo)*f(hi) <= 0 else 2.


def partition(x, t, method):
    # x is sorted. Both arms use exactly the same deterministic partition.
    if method == "quantile":
        return np.minimum(np.arange(len(x))*t//len(x), t-1)
    if method != "lloyd":
        raise ValueError("Unknown deterministic initialization.")
    centers = np.quantile(x, (np.arange(t)+.5)/t)
    old = None
    for _ in range(100):
        distances = (x[:, None]-centers)**2
        labels = distances.argmin(axis=1)
        counts = np.bincount(labels, minlength=t)
        for j in np.flatnonzero(counts == 0):
            residual = distances[np.arange(len(x)), labels].copy()
            residual[counts[labels] <= 1] = -np.inf
            index = int(np.argmax(residual))
            counts[labels[index]] -= 1
            labels[index] = j
            counts[j] += 1
        if old is not None and np.array_equal(labels, old):
            break
        centers = np.array([x[labels == j].mean() for j in range(t)])
        old = labels.copy()
    order = np.argsort(centers, kind="stable")
    inverse = np.empty(t, int)
    inverse[order] = np.arange(t)
    return inverse[labels]


In [ ]:
def copy_state(state):
    return {key: np.array(state[key], copy=True) for key in ("r", "y", "ga", "glb", "sa", "sb")}


class VariationalGGMM:
    def __init__(self, config=None):
        self.config = config or VGGMMConfig()

    def _empty_slot(self, state, k):
        # An unused b point estimate has no prior/likelihood contribution at Nk=0.
        # Canonical b=2 avoids exp(log(scale)) underflow from an old tiny shape.
        state["y"][k] = [0., np.log(self.config.mean_prior_sd), np.log(2.)]
        state["ga"][k] = self.config.precision_prior_shape
        state["glb"][k] = np.log(self.config.precision_prior_rate)

    def _profile_state(self, state, x):
        nk = state["r"].sum(axis=0)
        state["sa"], state["sb"] = stick_posterior(nk, self.config.concentration)
        for k in range(len(nk)):
            if nk[k] < 1e-12:
                self._empty_slot(state, k)
            else:
                _, state["ga"][k], state["glb"][k], _ = component_profile(
                    state["y"][k], x, state["r"][:, k], self.config)
        return state

    def _density(self, state):
        b = np.exp(state["y"][:, 2])
        loga = (state["glb"]-np.log(state["ga"]))/b + np.log(self.unit_)
        with np.errstate(over="raise", invalid="raise", under="ignore"):
            scales = np.exp(loga)
        weights, _ = stick_statistics(state["sa"], state["sb"])
        return Mixture(weights, state["y"][:, 0]*self.unit_+self.center_, scales, b)

    def _elbo(self, state, x):
        elogpdf = expected_log_components(x, state["y"], state["ga"], state["glb"])
        value = elbo_value(state["r"], elogpdf, state["sa"], state["sb"],
                           state["y"], state["ga"], state["glb"], self.config)
        if not np.isfinite(value):
            raise FloatingPointError("Nonfinite full ELBO.")
        return value-len(x)*np.log(self.unit_), elogpdf

    def _component_update(self, x, r, initial_y):
        nk = float(r.sum())
        old_value, *_ = component_profile(initial_y, x, r, self.config)
        best = [old_value, initial_y.copy()]
        rejections = 0

        def objective(params):
            nonlocal rejections
            try:
                value, *_ = component_profile(params, x, r, self.config)
                if value > best[0]:
                    best[:] = [value, params.copy()]
                return -value/max(nk, 1.)
            except (FloatingPointError, ValueError, OverflowError):
                rejections += 1
                return 1e100

        opt = minimize(objective, initial_y, method="L-BFGS-B",
            options=dict(maxiter=self.config.inner_max_iter, ftol=self.config.inner_ftol, maxls=30))
        _, ga, glb, _ = component_profile(best[1], x, r, self.config)
        return best[1], ga, glb, int(not opt.success), rejections, int(opt.nfev)

    def _initial_state(self, x, t, name):
        labels = partition(x, t, name)
        r = np.eye(t)[labels]
        y = np.zeros((t, 3))
        for k in range(t):
            group = x[labels == k]
            sd = max(float(group.std()), np.sqrt(np.finfo(float).eps))
            b = 2. if self.config.initial_shape_mode == "shape2" else moment_initial_shape(group)
            y[k] = [group.mean(), np.log(sd/np.sqrt(len(group))), np.log(b)]
        state = dict(r=r, y=y, ga=np.empty(t), glb=np.empty(t), sa=np.empty(t-1), sb=np.empty(t-1))
        for k in range(t):
            try:
                _, state["ga"][k], state["glb"][k], _ = component_profile(y[k], x, r[:, k], self.config)
            except (FloatingPointError, ValueError):
                y[k, 2] = np.log(2.)
                _, state["ga"][k], state["glb"][k], _ = component_profile(y[k], x, r[:, k], self.config)
        state["sa"], state["sb"] = stick_posterior(r.sum(axis=0), self.config.concentration)
        return state

    def _optimize(self, x, initial_state, name):
        cfg = self.config
        state = copy_state(initial_state)
        self._profile_state(state, x)
        begin = perf_counter()
        history = []
        stable, previous, failures, rejections, nfev = 0, None, 0, 0, 0
        converged, reason = False, "outer_max_iter"
        for iteration in range(cfg.max_iter+1):
            value, elogpdf = self._elbo(state, x)
            weights, elogw = stick_statistics(state["sa"], state["sb"])
            history.append(dict(iteration=iteration, elbo=value,
                active_K=int(np.sum(weights > cfg.active_threshold)), seconds=perf_counter()-begin))
            if previous is not None:
                gain = (value-previous)/len(x)
                if gain < -1e-7:
                    reason = "ELBO_decreased_numerical_failure"
                    break
                stable = stable+1 if abs(gain) < cfg.tol else 0
                if stable >= 3:
                    converged, reason = True, "ELBO_gain_per_observation"
                    break
            if iteration == cfg.max_iter:
                break
            previous = value
            logjoint = elogpdf+elogw  # All Kmax slots, including newly emptied slots.
            lognorm = logsumexp(logjoint, axis=1, keepdims=True)
            if not np.isfinite(lognorm).all():
                raise FloatingPointError("No finite expected density at an observation.")
            state["r"] = np.exp(logjoint-lognorm)
            state["sa"], state["sb"] = stick_posterior(state["r"].sum(axis=0), cfg.concentration)
            for k in range(state["r"].shape[1]):
                rk = state["r"][:, k]
                if float(rk.sum()) < 1e-12:
                    self._empty_slot(state, k)
                    continue
                y, ga, glb, fail, reject, evaluations = self._component_update(x, rk, state["y"][k])
                state["y"][k], state["ga"][k], state["glb"][k] = y, ga, glb
                failures += fail
                rejections += reject
                nfev += evaluations
        # Do not require a plug-in density at every iteration; ELBO controls inference.
        # The exported final density must nevertheless have representable parameters.
        self._density(state)
        return dict(name=name, state=state, history=history, elbo=history[-1]["elbo"],
            converged=converged, reason=reason, seconds=perf_counter()-begin,
            inner_nonconverged=failures, numerical_rejections=rejections, nfev=nfev)

    def _active_k(self, state):
        weights, _ = stick_statistics(state["sa"], state["sb"])
        return int(np.sum(weights > self.config.active_threshold))


    # Merge proposals and full-refit acceptance (same class / same VS Code cell).
    def _pair_pool(self, state, x):
        r = state["r"]
        nk = r.sum(axis=0)
        live = np.flatnonzero(nk > self.config.candidate_min_effective_count)
        if len(live) < 2:
            return []
        # Observation centers/spreads, not the SD of the posterior over a mean.
        centers = np.array([np.dot(r[:, k], x)/nk[k] for k in live])
        order = live[np.argsort(centers, kind="stable")]
        pairs = []
        for left, right in zip(order[:-1], order[1:]):
            i, j = sorted((int(left), int(right)))  # Never reorder the sticks.
            overlap = float(np.sqrt(r[:, i]*r[:, j]).sum()/np.sqrt(nk[i]*nk[j]))
            pairs.append(dict(i=i, j=j, overlap=overlap))
        return pairs

    def _merge_seed(self, incumbent, x, i, j):
        state = copy_state(incumbent)
        parents = np.exp(state["y"][[i, j], 2])
        state["r"][:, i] += state["r"][:, j]
        state["r"][:, j] = 0.
        # The total responsibility remains one for every observation.
        pooled = state["r"][:, i]
        nk = float(pooled.sum())
        mean = float(np.dot(pooled, x)/nk)
        variance = float(np.dot(pooled, (x-mean)**2)/nk)
        posterior_sd = np.sqrt(max(variance, np.finfo(float).eps)/max(nk, 1.))
        # Shape seeds affect only proposal initialization. The fitted shape stays free.
        shapes = np.unique(np.r_[2., parents])
        best, seed_failures = None, []
        for b in shapes:
            y0 = np.array([mean, np.log(posterior_sd), np.log(b)])
            try:
                y, ga, glb, fail, reject, nfev = self._component_update(x, pooled, y0)
                value, *_ = component_profile(y, x, pooled, self.config)
                if best is None or value > best[0]:
                    best = (value, y, ga, glb, fail, reject, nfev, float(b))
            except (FloatingPointError, ValueError, OverflowError) as exc:
                seed_failures.append(f"b={b:g}: {type(exc).__name__}: {exc}")
        if best is None:
            raise FloatingPointError(f"All merged-component seeds failed: {seed_failures}")
        _, state["y"][i], state["ga"][i], state["glb"][i], *info = best
        self._empty_slot(state, j)
        self._profile_state(state, x)
        seed_elbo, _ = self._elbo(state, x)
        return state, seed_elbo, best[-1], seed_failures

    def _merge_search(self, x, baseline):
        cfg = self.config
        incumbent = baseline
        logs, controls, checkpoints, accepted = [], [], [baseline], 0
        begin = perf_counter()
        stop_reason = "merge_round_budget_exhausted"
        for round_index in range(1, cfg.max_merge_rounds+1):
            # Give unmerged coordinate ascent another equal-budget continuation.
            # Compare proposals against the improved control, not an underoptimized baseline.
            try:
                control = self._optimize(x, incumbent["state"], f"continuation-{round_index}")
                control_gain = (control["elbo"]-incumbent["elbo"])/len(x)
                took_control = bool(control["converged"] and control_gain > 0.)
                controls.append(dict(Round=round_index, Before_ELBO=incumbent["elbo"],
                    After_ELBO=control["elbo"], Gain_per_observation=control_gain,
                    Converged=control["converged"], Used=took_control,
                    Inner_nonconverged=control["inner_nonconverged"], Error=""))
                if took_control:
                    incumbent = control
            except (FloatingPointError, ValueError, OverflowError) as exc:
                controls.append(dict(Round=round_index, Used=False, Converged=False,
                    Error=f"{type(exc).__name__}: {exc}"))
            pool = self._pair_pool(incumbent["state"], x)
            if not pool:
                stop_reason = "fewer_than_two_numerically_occupied_slots"
                break
            prepared = []
            for pair in pool:
                try:
                    seed, value, seed_b, seed_errors = self._merge_seed(
                        incumbent["state"], x, pair["i"], pair["j"])
                    prepared.append(dict(**pair, state=seed, seed_elbo=value, seed_b=seed_b,
                                         seed_errors=seed_errors))
                except (FloatingPointError, ValueError, OverflowError) as exc:
                    logs.append(dict(Round=round_index, Pair_i=pair["i"], Pair_j=pair["j"],
                        Accepted=False, Status="seed_failed", Error=f"{type(exc).__name__}: {exc}"))
            # Full seed ELBO ranks proposals. A low seed ELBO is not itself a rejection.
            prepared.sort(key=lambda item: (-item["seed_elbo"], -item["overlap"], item["i"], item["j"]))
            if not prepared:
                stop_reason = "all_merge_seeds_failed"
                break
            any_accepted = False
            for rank, proposal in enumerate(prepared[:cfg.max_candidates_per_round], start=1):
                before = incumbent["elbo"]
                before_k = self._active_k(incumbent["state"])
                entry = dict(Round=round_index, Rank=rank, Pair_i=proposal["i"], Pair_j=proposal["j"],
                    Pair_overlap=proposal["overlap"], Seed_b=proposal["seed_b"],
                    Seed_errors="; ".join(proposal["seed_errors"]),
                    Seed_ELBO=proposal["seed_elbo"], Before_ELBO=before,
                    Before_active_K=before_k, Pool_size=len(pool),
                    Prepared_candidates=len(prepared), Accepted=False, Error="")
                try:
                    candidate = self._optimize(x, proposal["state"],
                        f"merge-{round_index}-{proposal['i']}-{proposal['j']}")
                    gain = (candidate["elbo"]-before)/len(x)
                    after_k = self._active_k(candidate["state"])
                    take = bool(candidate["converged"] and gain > cfg.merge_gain_per_observation)
                    entry.update(After_ELBO=candidate["elbo"], Gain_per_observation=gain,
                        After_active_K=after_k, Refit_converged=candidate["converged"],
                        Refit_stop_reason=candidate["reason"], Refit_iterations=len(candidate["history"])-1,
                        Inner_nonconverged=candidate["inner_nonconverged"],
                        Refit_s=candidate["seconds"], Accepted=take,
                        Status="accepted" if take else "rejected")
                    if take:
                        incumbent, any_accepted = candidate, True
                        accepted += 1
                        checkpoints.append(candidate)
                except (FloatingPointError, ValueError, OverflowError) as exc:
                    entry.update(Status="refit_failed", Error=f"{type(exc).__name__}: {exc}")
                logs.append(entry)
                if entry["Accepted"]:
                    print(f"    merge ({proposal['i']}, {proposal['j']}): K {before_k}->{after_k}, "
                          f"ELBO/n +{gain:.6g}", flush=True)
                    break  # First improving proposal; rebuild the pool from the new state.
            if not any_accepted:
                stop_reason = ("candidate_budget_exhausted_without_improvement"
                    if len(prepared) > cfg.max_candidates_per_round
                    else "no_improvement_in_all_prepared_adjacent_pairs")
                break
        self.merge_log_, self.continuation_log_, self.merge_checkpoints_ = logs, controls, checkpoints
        self.accepted_merges_, self.merge_stop_reason_ = accepted, stop_reason
        self.merge_seconds_ = perf_counter()-begin
        return incumbent

    def fit(self, X):
        cfg = self.config
        started = perf_counter()
        x = np.array(X, dtype=float, copy=True)
        if x.ndim == 2 and x.shape[1] == 1:
            x = x[:, 0]
        if x.ndim != 1 or len(x) < 2 or not np.isfinite(x).all() or np.std(x) <= 0:
            raise ValueError("X must be a finite nonconstant continuous column.")
        integer_fields = (cfg.max_components, cfg.max_iter, cfg.inner_max_iter,
                          cfg.max_merge_rounds, cfg.max_candidates_per_round)
        if (any(not isinstance(v, (int, np.integer)) or isinstance(v, (bool, np.bool_)) or v < 1
                for v in integer_fields)
            or not cfg.starts or any(s not in ("quantile", "lloyd") for s in cfg.starts)
            or cfg.initial_shape_mode not in ("moment", "shape2")
            or not 0 <= cfg.active_threshold < 1
            or any(not np.isfinite(v) or v <= 0 for v in (cfg.concentration, cfg.tol,
                cfg.mean_prior_sd, cfg.precision_prior_shape, cfg.precision_prior_rate,
                cfg.inner_ftol, cfg.merge_gain_per_observation, cfg.candidate_min_effective_count))):
            raise ValueError("Invalid VGGMM config.")
        self.center_, self.unit_, self.raw_x_ = float(x.mean()), float(x.std()), x.copy()
        z = np.sort((x-self.center_)/self.unit_)
        self.standardized_x_ = z
        t = min(cfg.max_components, len(np.unique(x)), len(x))
        self.runs_, self.failed_starts_ = [], []
        for name in cfg.starts:
            try:
                state = self._initial_state(z, t, name)
                self.runs_.append(self._optimize(z, state, name))
            except (FloatingPointError, ValueError, OverflowError) as exc:
                self.failed_starts_.append(dict(start=name, error_type=type(exc).__name__, error=str(exc)))
        if not self.runs_:
            raise RuntimeError(f"All starts failed: {self.failed_starts_}")
        # Select from converged starts if available; report a fallback explicitly.
        valid = [run for run in self.runs_ if run["converged"]]
        self.baseline_selection_fallback_ = not bool(valid)
        baseline = max(valid or self.runs_, key=lambda run: run["elbo"])
        self.baseline_run_ = baseline
        self.baseline_density_ = self._density(baseline["state"])
        self.baseline_seconds_ = perf_counter()-started
        self.merge_log_, self.continuation_log_, self.merge_checkpoints_ = [], [], [baseline]
        self.accepted_merges_, self.merge_seconds_, self.merge_stop_reason_ = 0, 0., "disabled"
        final = self._merge_search(z, baseline) if cfg.enable_merge else baseline
        self.final_run_, self.posterior_state_ = final, copy_state(final["state"])
        self.full_density_ = self._density(final["state"])
        self.weights_ = self.full_density_.weights.copy()
        keep = self.weights_ > cfg.active_threshold
        if not keep.any():
            raise RuntimeError("No active component at the reporting threshold.")
        self.density_ = Mixture(self.weights_[keep]/self.weights_[keep].sum(),
            self.full_density_.means[keep], self.full_density_.scales[keep], self.full_density_.shapes[keep])
        self.n_components_, self.full_k_ = int(keep.sum()), t
        self.converged_, self.termination_reason_ = final["converged"], final["reason"]
        self.fit_seconds_ = perf_counter()-started
        self.history_ = final["history"]
        if not self.converged_:
            warnings.warn("Selected fit did not satisfy the outer convergence criterion.", RuntimeWarning)
        if self.failed_starts_:
            warnings.warn(f"{len(self.failed_starts_)} initial start(s) failed; see failed_starts_.", RuntimeWarning)
        return self


In [ ]:
def active_view(full, threshold):
    keep = full.weights > threshold
    retained = float(full.weights[keep].sum())
    excluded = float(full.weights[~keep].sum())
    density = (Mixture(full.weights[keep]/retained, full.means[keep], full.scales[keep],
                      full.shapes[keep]) if keep.any() else None)
    return density, int(keep.sum()), excluded


_LEGENDRE = {order: leggauss(order) for order in (32, 64)}


def density_errors(truth, fitted, tail=1e-9, atol=1e-7, rtol=1e-5, max_refinement=16):
    """Adaptive IAE/ISE quadrature, with estimated quadrature errors and tail bounds."""
    probabilities = np.array([tail, 1e-6, 1e-4, .001, .01, .1, .25, .5,
                              .75, .9, .99, .999, 1-1e-4, 1-1e-6, 1-tail])
    points = np.unique(np.concatenate([gennorm.ppf(probabilities[:, None], m.shapes,
        loc=m.means, scale=m.scales).ravel() for m in (truth, fitted)]))
    if not np.isfinite(points).all():
        raise FloatingPointError("Evaluation quantiles cannot be represented.")
    left, right = points[:-1], points[1:]
    n_panels = len(left)
    depth = np.zeros(n_panels, int)
    total, error = np.zeros(2), np.zeros(2)
    unresolved = 0

    def panel(l, r, order):
        nodes, weights = _LEGENDRE[order]
        mid, half = l/2+r/2, (r-l)/2
        xx = mid[:, None]+half[:, None]*nodes
        delta = (fitted.pdf(xx.ravel())-truth.pdf(xx.ravel())).reshape(xx.shape)
        return np.c_[np.abs(delta)@weights, delta**2@weights]*half[:, None]

    while len(left):
        low, high = panel(left, right, 32), panel(left, right, 64)
        err = np.abs(high-low)
        good = np.all(err <= atol/n_panels/(2.**depth[:, None])+rtol*np.abs(high), axis=1)
        stopped = depth >= max_refinement
        accept = good | stopped
        total += high[accept].sum(axis=0)
        error += err[accept].sum(axis=0)
        unresolved += int(np.sum(stopped & ~good))
        ll, rr, dd = left[~accept], right[~accept], depth[~accept]
        mid = ll/2+rr/2
        left, right, depth = np.r_[ll, mid], np.r_[mid, rr], np.r_[dd+1, dd+1]
    lo, hi = points[0], points[-1]

    def omitted(model):
        return float(np.clip(model.cdf([lo])[0]+1-model.cdf([hi])[0], 0, 1))

    def peak(model):
        return float(np.sum(model.weights*np.exp(-np.log(2)-np.log(model.scales)
                                                -gammaln(1+1/model.shapes))))

    tt, tf = omitted(truth), omitted(fitted)
    return dict(IAE=float(total[0]), ISE=float(total[1]),
        IAE_quad_error=float(error[0]), ISE_quad_error=float(error[1]),
        IAE_tail_upper=tt+tf, ISE_tail_upper=peak(truth)*tt+peak(fitted)*tf,
        Quadrature_OK=bool(unresolved == 0 and np.all(error <= atol+rtol*np.abs(total))))


def check_mathematics():
    # At b=2, E|x-U|² has an elementary closed form independent of hyp1f1.
    xx, mean, sd = np.array([-3., -.1, 0., 2., 10.]), .4, .7
    np.testing.assert_allclose(np.exp(normal_log_abs_moment(xx, mean, np.log(sd), 2.)),
                               (xx-mean)**2+sd**2, rtol=2e-13)
    # Check the profiled q(tau) contribution against the unprofiled likelihood - KL.
    cfg = BASE_CONFIG
    r = np.array([.1, .7, .4, .8, .2])
    y = np.array([mean, np.log(sd), np.log(1.7)])
    profiled, a, lb, b = component_profile(y, xx, r, cfg)
    expected = expected_log_components(xx, y[None, :], np.array([a]), np.array([lb]))[:, 0]
    expanded = r@expected-normal_kl(mean, np.log(sd), cfg.mean_prior_sd)
    expanded -= gamma_kl(a, lb, cfg.precision_prior_shape, cfg.precision_prior_rate)
    np.testing.assert_allclose(profiled, expanded, rtol=2e-13, atol=1e-12)
    for nk in (np.array([100., 0., 40., 2.]), np.array([5.])):
        sa, sb = stick_posterior(nk, cfg.concentration)
        w, elogw = stick_statistics(sa, sb)
        np.testing.assert_allclose(w.sum(), 1., atol=1e-14)
        assert np.all(elogw <= np.log(w)+1e-10)
    print("Math checks passed: normal moment, Gamma profile/ELBO, stick expectations.")


check_mathematics()


In [ ]:
def fit_vgm(x, cap, cfg):
    estimator = BayesianGaussianMixture(n_components=cap, covariance_type="full",
        weight_concentration_prior_type="dirichlet_process",
        weight_concentration_prior=cfg.concentration, init_params="kmeans", n_init=1,
        max_iter=1500, tol=1e-3, reg_covar=1e-6, random_state=27)
    started = perf_counter()
    with threadpool_limits(limits=1):
        estimator.fit(np.array(x, copy=True).reshape(-1, 1))
    full = Mixture(estimator.weights_/estimator.weights_.sum(), estimator.means_[:, 0],
        np.sqrt(2*estimator.covariances_[:, 0, 0]), np.full(cap, 2.))
    return dict(method="VGM", cap=cap, full=full, fit_seconds=perf_counter()-started,
        converged=bool(estimator.converged_), iterations=int(estimator.n_iter_),
        chosen_start="kmeans", elbo=np.nan, chosen_inner_nonconverged=0,
        all_inner_nonconverged=0, failed_starts=0,
        stop_reason="tolerance" if estimator.converged_ else "iteration_limit", model=None)


def show_table(frame):
    try:
        from IPython import get_ipython
        from IPython.display import display
        if get_ipython() is not None:
            display(frame)
            return
    except ImportError:
        pass
    print(frame.to_string(index=False, float_format=lambda v: f"{v:.7g}"))


def evaluate_density(method, cap, full, x, reference, cfg, run, fit_s, accepted_merges=0,
                     merge_stop_reason="disabled", failed_starts=0):
    active, k, excluded = active_view(full, cfg.active_threshold)
    if active is None:
        raise RuntimeError("No active density at the reporting threshold.")
    full_errors, errors = density_errors(reference, full), density_errors(reference, active)
    row = dict(Method=method, Max_K=cap, Active_K=k, Full_slots=full.k,
        Excluded_mass=excluded, IAE=errors["IAE"], ISE=errors["ISE"],
        IAE_full=full_errors["IAE"], ISE_full=full_errors["ISE"],
        Quadrature_OK=bool(errors["Quadrature_OK"] and full_errors["Quadrature_OK"]),
        ELBO_per_observation=run["elbo"]/len(x), LL_full_per_observation=float(full.logpdf(x).mean()),
        Converged=run["converged"], Inner_nonconverged=run["inner_nonconverged"],
        Accepted_merges=accepted_merges, Merge_stop_reason=merge_stop_reason,
        Fit_s=fit_s, Failed_initial_starts=failed_starts,
        IAE_quad_error=errors["IAE_quad_error"], ISE_quad_error=errors["ISE_quad_error"],
        IAE_tail_upper=errors["IAE_tail_upper"], ISE_tail_upper=errors["ISE_tail_upper"])
    return row, active


def run_cap(x, cap, cfg, reference, output_dir, include_vgm=True):
    current = replace(cfg, max_components=cap)
    print(f"\nmax_K={cap}: baseline fit + merge search ({cfg.initial_shape_mode} initialization)", flush=True)
    with threadpool_limits(limits=1):
        model = VariationalGGMM(current).fit(x)
    rows, components, traces, starts = [], [], [], []
    records = []
    stages = [("VGGMM-base", model.baseline_run_, model.baseline_density_, model.baseline_seconds_, 0, "disabled"),
              ("VGGMM-merge", model.final_run_, model.full_density_, model.fit_seconds_,
               model.accepted_merges_, model.merge_stop_reason_)]
    for method, run, full, seconds, accepted, stop_reason in stages:
        row, active = evaluate_density(method, cap, full, x, reference, cfg, run, seconds,
            accepted_merges=accepted, merge_stop_reason=stop_reason, failed_starts=len(model.failed_starts_))
        rows.append(row)
        records.append(dict(method=method, cap=cap, full=full, active=active))
        nk = run["state"]["r"].sum(axis=0)
        for k in range(full.k):
            components.append(dict(Method=method, Max_K=cap, Component=k, Weight=full.weights[k],
                Mean=full.means[k], Scale=full.scales[k], Shape=full.shapes[k], Sigma=full.sigmas[k],
                Nk=nk[k], Active=bool(full.weights[k] > cfg.active_threshold)))
        print(f"  {method}: Active_K={row['Active_K']}, ELBO/n={row['ELBO_per_observation']:.7f}, "
              f"IAE={row['IAE']:.7f}, merges={accepted}", flush=True)
    for run in model.runs_:
        starts.append(dict(Max_K=cap, Start=run["name"], Selected=bool(run is model.baseline_run_),
            Converged=run["converged"], ELBO_per_observation=run["elbo"]/len(x),
            Active_K=model._active_k(run["state"]), Inner_nonconverged=run["inner_nonconverged"],
            Stop_reason=run["reason"], Fit_s=run["seconds"]))
    for stage, run in [("baseline", model.baseline_run_)] + [
            (f"accepted-{i}", item) for i, item in enumerate(model.merge_checkpoints_[1:], start=1)]:
        for item in run["history"]:
            traces.append(dict(Max_K=cap, Stage=stage, **item))
    if include_vgm:
        rec = fit_vgm(x, cap, cfg)
        fake_run = dict(elbo=np.nan, converged=rec["converged"], inner_nonconverged=0)
        row, active = evaluate_density("VGM", cap, rec["full"], x, reference, cfg, fake_run,
                                       rec["fit_seconds"])
        rows.append(row)
        records.append(dict(method="VGM", cap=cap, full=rec["full"], active=active))
        full = rec["full"]
        for k in range(full.k):
            components.append(dict(Method="VGM", Max_K=cap, Component=k, Weight=full.weights[k],
                Mean=full.means[k], Scale=full.scales[k], Shape=full.shapes[k], Sigma=full.sigmas[k],
                Nk=np.nan, Active=bool(full.weights[k] > cfg.active_threshold)))
    tables = dict(metrics=pd.DataFrame(rows), components=pd.DataFrame(components),
        starts=pd.DataFrame(starts), traces=pd.DataFrame(traces),
        merge_log=pd.DataFrame(model.merge_log_), continuation_log=pd.DataFrame(model.continuation_log_),
        failed_starts=pd.DataFrame(model.failed_starts_, columns=["start", "error_type", "error"]))
    cap_dir = output_dir / f"cap_{cap}"
    cap_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(cap_dir / f"{name}.csv", index=False)
    # Full posterior states support audits/warm refits. No hidden fit cache is used.
    arrays = {}
    for name, run in (("baseline", model.baseline_run_), ("final", model.final_run_)):
        arrays.update({f"{name}_{key}": value for key, value in run["state"].items()})
    arrays.update(center=model.center_, unit=model.unit_)
    np.savez_compressed(cap_dir / "posterior_states.npz", **arrays)
    (cap_dir / "summary.json").write_text(json.dumps(dict(config=asdict(current),
        baseline_selection_fallback=model.baseline_selection_fallback_,
        accepted_merges=model.accepted_merges_, merge_stop_reason=model.merge_stop_reason_,
        baseline_seconds=model.baseline_seconds_, merge_seconds=model.merge_seconds_,
        full_slots=model.full_k_, threshold=cfg.active_threshold), indent=2), encoding="utf-8")
    return model, tables, records


def combined_tables(cap_tables):
    tables = {}
    for name in ("metrics", "components", "starts", "traces", "merge_log", "continuation_log", "failed_starts"):
        frames = [table[name].assign(Max_K=cap) if "Max_K" not in table[name].columns else table[name]
                  for cap, table in cap_tables if not table[name].empty]
        tables[name] = (pd.concat(frames, ignore_index=True) if frames else
                        pd.DataFrame(columns=cap_tables[0][1][name].columns))
    rows = []
    for cap, group in tables["metrics"].groupby("Max_K", sort=True):
        by_method = group.set_index("Method")
        base, merge = by_method.loc["VGGMM-base"], by_method.loc["VGGMM-merge"]
        logs = tables["merge_log"]
        controls = tables["continuation_log"]
        accepted_gain = 0.
        continuation_gain = 0.
        if not logs.empty:
            chosen = logs[(logs["Max_K"] == cap) & logs["Accepted"].fillna(False).astype(bool)]
            if not chosen.empty:
                accepted_gain = float(chosen["Gain_per_observation"].sum())
        if not controls.empty:
            used = controls[(controls["Max_K"] == cap) & controls["Used"].fillna(False).astype(bool)]
            if not used.empty:
                continuation_gain = float(used["Gain_per_observation"].sum())
        rows.append(dict(Max_K=cap, Base_active_K=int(base["Active_K"]),
            Merge_active_K=int(merge["Active_K"]), Base_IAE=base["IAE"], Merge_IAE=merge["IAE"],
            Base_ISE=base["ISE"], Merge_ISE=merge["ISE"],
            Total_delta_ELBO_per_observation=merge["ELBO_per_observation"]-base["ELBO_per_observation"],
            Accepted_merge_ELBO_gain_per_observation=accepted_gain,
            Continuation_ELBO_gain_per_observation=continuation_gain,
            Accepted_merges=int(merge["Accepted_merges"]), Merge_stop_reason=merge["Merge_stop_reason"]))
    tables["paired"] = pd.DataFrame(rows)
    return tables


In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--caps", nargs="+", type=int, default=list(CAPS))
    parser.add_argument("--initial-shape", choices=("moment", "shape2"), default=INITIAL_SHAPE_MODE)
    parser.add_argument("--no-vgm", action="store_true")
    parser.add_argument("--no-show", action="store_true")
    parser.add_argument("--output", type=Path)
    args, _ = parser.parse_known_args()
    caps = tuple(args.caps)
    if not caps or len(caps) != len(set(caps)) or any(k < 1 for k in caps):
        raise ValueError("caps must be distinct positive integers.")
    cfg = replace(BASE_CONFIG, initial_shape_mode=args.initial_shape)
    try:
        script_root = Path(__file__).resolve().parent
    except NameError:
        script_root = Path.cwd()
    OUTPUT_DIR = (args.output or script_root / "results") / datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    models, cap_tables, records = {}, [], []
    for cap in caps:
        model, one_table, one_records = run_cap(X, cap, cfg, truth, OUTPUT_DIR,
                                               INCLUDE_VGM and not args.no_vgm)
        models[cap] = model
        cap_tables.append((cap, one_table))
        records.extend(one_records)
    tables = combined_tables(cap_tables)
    for name, table in tables.items():
        table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    show_table(tables["paired"])
    np.savez_compressed(OUTPUT_DIR / "sample.npz", X=X)
    metadata = dict(n=N, seed=SAMPLE_SEED, sample_sha256=SAMPLE_SHA256, truth=truth.payload(),
        config=asdict(cfg), caps=caps, shape_is_free=True, fitting_uses_truth=False,
        same_cap_for_ELBO_acceptance=True, slot_order_preserved=True,
        empty_slots_may_reactivate=True, control_continuation_before_each_merge_round=True,
        acceptance="converged full refit and ELBO gain/n > merge_gain_per_observation",
        proposal_pool="adjacent occupied components by weighted observation center; no slot permutation",
        proposal_ranking="seed full ELBO; responsibility overlap breaks ties",
        search="first improving among ranked candidates up to per-round budget",
        density_kind="posterior-parameter plug-in GGMM, not full posterior predictive",
        numerical_empty_reset="Nk<1e-12: q(mu)/q(tau) prior, b point estimate=2 in base and merge",
        python=platform.python_version(), versions={name: version(name) for name in
            ("numpy", "scipy", "pandas", "matplotlib", "scikit-learn", "threadpoolctl")})
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2),
                                              encoding="utf-8")
    assert hashlib.sha256(np.asarray(X, dtype="<f8").tobytes()).hexdigest() == SAMPLE_SHA256
    print(f"Saved: {OUTPUT_DIR}", flush=True)


In [ ]:
if __name__ == "__main__":
    styles = {"VGGMM-base": ("#2865b0", "--"), "VGGMM-merge": ("#c43c39", "-"),
              "VGM": ("#2c8b66", ":")}
    metrics = tables["metrics"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
    for method, part in metrics.groupby("Method", sort=False):
        color, style = styles[method]
        for ax, metric in zip(axes.flat, ("Active_K", "IAE", "ISE", "ELBO_per_observation")):
            if method == "VGM" and metric == "ELBO_per_observation":
                continue
            ax.plot(part.Max_K, part[metric], marker="o", color=color, ls=style, label=method)
    axes[0, 0].plot(caps, caps, ":", color="gray", label="K = cap")
    axes[0, 0].axhline(truth.k, color="black", ls=":", label="True K (evaluation only)")
    for ax, metric in zip(axes.flat, ("Active K", "IAE", "ISE", "ELBO / observation")):
        ax.set(xlabel="Maximum components", ylabel=metric, xticks=caps)
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.suptitle(f"VGGMM merge/refit: n={N}, seed={SAMPLE_SEED}; same cap/priors")
    fig.savefig(OUTPUT_DIR / "cap_comparison.png", dpi=180)
    if not args.no_show:
        plt.show()
    plt.close(fig)
    ncols, nrows = min(3, len(caps)), int(np.ceil(len(caps)/3))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.8*nrows), squeeze=False,
                             sharex=True, sharey=True, layout="constrained")
    grid_lo = gennorm.ppf(1e-5, truth.shapes, loc=truth.means, scale=truth.scales).min()
    grid_hi = gennorm.ppf(1-1e-5, truth.shapes, loc=truth.means, scale=truth.scales).max()
    grid = np.linspace(min(grid_lo, X.min()), max(grid_hi, X.max()), 2200)
    for ax, cap in zip(axes.flat, caps):
        ax.hist(X, bins=70, density=True, color="#dce2e8", edgecolor="white", linewidth=.3)
        ax.plot(grid, truth.pdf(grid), "k:", lw=1.7, label="True density")
        for rec in records:
            if rec["cap"] == cap:
                color, style = styles[rec["method"]]
                ax.plot(grid, rec["active"].pdf(grid), color=color, ls=style, lw=1.8,
                        label=f"{rec['method']}, K={rec['active'].k}")
        ax.set(title=f"max_K={cap}", xlabel="x", ylabel="Density")
        ax.legend(fontsize=8)
        ax.grid(alpha=.15)
    for ax in list(axes.flat)[len(caps):]:
        ax.set_visible(False)
    fig.suptitle("Same sample; ordinary variational fit vs accepted merge/refit result")
    fig.savefig(OUTPUT_DIR / "density_comparison.png", dpi=180)
    if not args.no_show:
        plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for cap in caps:
        model = models[cap]
        accepted = [model.baseline_run_] + model.merge_checkpoints_[1:]
        axes[0].plot(range(len(accepted)), [model._active_k(run["state"]) for run in accepted],
                     "o-", label=f"max_K={cap}")
        axes[1].plot(range(len(accepted)), [run["elbo"]/N for run in accepted],
                     "o-", label=f"max_K={cap}")
    for ax, metric in zip(axes, ("Active K", "ELBO / observation")):
        ax.set(xlabel="Accepted merge count", ylabel=metric)
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.suptitle("Accepted moves (unmerged continuation gains are logged separately)")
    fig.savefig(OUTPUT_DIR / "accepted_moves.png", dpi=180)
    if not args.no_show:
        plt.show()
    plt.close(fig)
    print("Full-state ELBO decides acceptance; active_K/IAE/ISE/true K never decide acceptance.")
    print("Active_K may stay unchanged or increase after full refit. Search budget limits are logged.")
